# Notebook 3: Optimized SARIMAX and Hybrid Refactor

This notebook is a clean refactor of the existing forecasting setup. It removes exploratory clutter, standardizes wording, and organizes the workflow into a production-style pipeline.

The default strategy is a hybrid comparison:

- Tune pure SARIMAX for short monthly horizons.
- Optionally compare Prophet and a simple Prophet plus SARIMAX ensemble.
- Select the best validation performer and expose the same `run_forecast(model, months_horizon)` interface used by the other notebooks.

## Runtime Configuration

For SARIMAX, short horizons are usually most reliable. This notebook defaults to 6 months while still allowing 1 to 12 for UI compatibility.

In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")

DATASET_PATH = "dataset/credit_card_transactions.csv"
TIMESTAMP_COL = "trans_date_trans_time"
MONTHS_HORIZON = 6
VALIDATION_MONTHS = 3

MONTHS_HORIZON = 6
VALIDATION_MONTHS = 3
MODEL_STRATEGY = "hybrid"  # Use "sarimax" for pure SARIMAX only.

## 1. Data Ingestion & Preprocessing

This step converts raw transaction events into a complete monthly transaction-count series. It handles missing or invalid timestamps and removes the partial final month before modeling.

In [ ]:
from src.time_series_common import MonthlyDataConfig, load_monthly_transaction_volume

config = MonthlyDataConfig(csv_path=DATASET_PATH, timestamp_col=TIMESTAMP_COL)
monthly_series, profile = load_monthly_transaction_volume(config)

print(f"Rows processed: {profile.row_count:,}")
print(f"Invalid timestamps: {profile.invalid_timestamp_count:,}")
print(f"Complete monthly observations: {len(monthly_series)}")
print(f"Last complete month: {profile.last_complete_month}")

display(monthly_series.rename("monthly_transaction_volume").to_frame())

## 2. Feature Engineering / Stationarity Checks

SARIMAX requires attention to stationarity. A stationary series has a relatively stable mean and variance over time. If the series is not stationary, differencing is commonly used.

This pipeline runs the Augmented Dickey-Fuller test and tunes SARIMAX orders that include differenced and non-differenced candidates.

In [ ]:
from src.sarimax_hybrid_pipeline import run_adfuller_test

stationarity_report = run_adfuller_test(monthly_series)
display(pd.DataFrame([stationarity_report]))

if stationarity_report["is_stationary_at_5pct"]:
    print("ADF result: the monthly series is stationary at the 5% significance level.")
else:
    print("ADF result: the monthly series is not stationary at the 5% significance level. Differenced SARIMAX candidates will be evaluated.")

## 3. Model Training & Parameter Tuning

The refactored pipeline tunes compact SARIMAX order candidates using validation MAE and RMSE. If `MODEL_STRATEGY = "hybrid"`, it also compares Prophet and a simple average ensemble of Prophet and SARIMAX.

This keeps the workflow clean and decision-oriented:

1. Tune SARIMAX.
2. Evaluate optional Prophet component.
3. Evaluate optional ensemble.
4. Select the validation winner.
5. Refit on complete monthly history.

In [ ]:
from src.sarimax_hybrid_pipeline import train_sarimax_hybrid_pipeline

refactor_model = train_sarimax_hybrid_pipeline(
    monthly_series=monthly_series,
    profile=profile,
    validation_months=VALIDATION_MONTHS,
    model_strategy=MODEL_STRATEGY,
)

print(f"Selected model: {refactor_model['model_name']}")
print(f"Best SARIMAX order: {refactor_model['best_order']}")

display(refactor_model["sarimax_tuning_metrics"])
display(refactor_model["comparison_metrics"])

## 4. Evaluation Metrics

The selected model is chosen using validation metrics:

- **MAE:** average monthly transaction-count error.
- **RMSE:** error metric that emphasizes larger misses.
- **MAPE:** percentage error, useful for business interpretation.

For this refactor, the most important quality check is whether tuned SARIMAX or the hybrid model improves over simpler alternatives without adding unnecessary complexity.

In [ ]:
best_refactor_metrics = refactor_model["comparison_metrics"].iloc[0]

print(f"Validation MAE: {best_refactor_metrics['MAE']:,.0f} transactions/month")
print(f"Validation RMSE: {best_refactor_metrics['RMSE']:,.0f} transactions/month")
print(f"Validation MAPE: {best_refactor_metrics['MAPE']:.2f}%")

## 5. Interactive Forecast Inference

This cell exposes the same inference interface as the other notebooks:

```python
forecast_df, forecast_fig = run_forecast(model, months_horizon)
```

The output is immediately usable in Streamlit.

In [ ]:
from src.sarimax_hybrid_pipeline import run_forecast

forecast_df, forecast_fig = run_forecast(refactor_model, MONTHS_HORIZON)

display(forecast_df)
forecast_fig.show()

## 6. Model Export

The export cell writes the selected SARIMAX/hybrid bundle to disk. The artifact includes fitted model objects, tuning metrics, comparison metrics, the ADF report, and training metadata.

In [ ]:
from src.sarimax_hybrid_pipeline import export_sarimax_hybrid_model

EXPORT_MODEL = True

if EXPORT_MODEL:
    artifact_path = export_sarimax_hybrid_model(refactor_model, "artifacts/sarimax_hybrid_model.pkl")
    print(f"Model artifact written to: {artifact_path}")

## Developer Insights

### Metric Interpretation

For monthly transaction volume, MAE is the clearest metric: it says how many transactions the forecast misses by in a typical month. RMSE helps identify whether the model has occasional large errors. MAPE translates the miss into a percentage of actual monthly volume, which is useful when communicating model quality to non-technical stakeholders.

For SARIMAX, pay close attention to whether validation error is stable across months. A low average MAE can hide one severe miss. If RMSE is much larger than MAE, the model may be fragile. For a hybrid model, accuracy is good only if the ensemble improves validation metrics or reduces large misses; otherwise pure SARIMAX is easier to explain and maintain.

### Model Retraining Strategy

Retrain after each new complete month is added. Recreate the monthly series, rerun the ADF test, retune SARIMAX orders, rerun the hybrid comparison if Prophet is enabled, and export a fresh artifact with the selected model and metrics.

For horizons up to 6 months, monitor whether SARIMAX remains competitive. If validation MAE deteriorates for two or more retraining cycles, inspect raw transaction ingestion, confirm the final month is complete, and compare against the Prophet and ML notebooks. Keep a small model registry table with artifact path, training date range, selected order, MAE, RMSE, MAPE, and production approval status.